# 1. Transformation Strategy

## 1.1 Objectives

The main objectives of this notebook are:

- load the raw FIPE snapshot without modifying the source;
- validate the expected raw-data assumptions;
- standardize fields with known inconsistencies;
- preserve business semantics identified during profiling;
- validate the transformed dataset;
- export a processed dataset for downstream analytical use.

## 1.2 Design Principles

The transformation process follows these principles:

- raw data must remain immutable;
- transformations must be deterministic and reproducible;
- business rules must be explicit;
- invalid records must be detected rather than silently corrected;
- `valor_centavos` is the canonical monetary field;
- `ano_modelo = NULL` must be preserved when associated with `zero_km = True`;
- the validated snapshot grain must remain unique;
- transformed data must comply with the documented data quality rules.

# 2. Imports and Configuration

In [15]:
from pathlib import Path
import pandas as pd
import re

In [13]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT/'data'/'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT/'data'/'processed'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [19]:
assert PROJECT_ROOT.exists(), "Project root directory not found."
assert RAW_DATA_DIR.exists(), "Raw data directory not found."
assert PROCESSED_DATA_DIR.exists(), "Processed data directory not found."

# 3. Data Loading

## 3.1 Raw Input

The raw source is loaded without applying transformations.

This preserves the original dataset and allows all changes to be explicitly tracked in the transformation pipeline.

In [29]:
raw_file = RAW_DATA_DIR / "fipex_prices_2026_09.parquet"

if not raw_file.exists():
    raise FileNotFoundError(f"Raw file not found: {raw_file}")

df_raw = pd.read_parquet(raw_file)

print(f"Source: {raw_file.name}")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

Source: fipex_prices_2026_09.parquet
Rows: 51,012
Columns: 12


In [34]:
df_raw.sample(20)

,tipo_veiculo,codigo_fipe,nome_modelo,nome_marca,nome_combustivel,sigla_combustivel,ano_modelo,zero_km,valor_centavos,valor_formatado,mes_referencia,ano_referencia
13550,carro,003059-7,Royale GL 1.8/1.8i 2p e 4p,Ford,Gasolina,g,1992.0,False,512400,"R$ 5.124,00",9,2026
26270,caminhão,509209-4,1420 2p (diesel),MERCEDES-BENZ,Diesel,d,2005.0,False,10291000,"R$ 102.910,00",9,2026
7351,carro,043002-1,Engesa 4x4 4.0 Diesel,Engesa,Diesel,d,1994.0,False,4636500,"R$ 46.365,00",9,2026
6553,moto,861021-5,HORIZON 150,DAFRA,Gasolina,g,2022.0,False,1218700,"R$ 12.187,00",9,2026
20970,caminhão,506039-7,EUROCARGO CAVALLINO 450-E32 T 2p (dies.),IVECO,Diesel,d,2010.0,False,11273200,"R$ 112.732,00",9,2026
21420,caminhão,506090-7,TECTOR 240E28 6x2 2p (dies.)(E5),IVECO,Diesel,d,2014.0,False,17435000,"R$ 174.350,00",9,2026
18459,moto,811171-5,PCX 160,HONDA,Gasolina,g,2024.0,False,1922400,"R$ 19.224,00",9,2026
47304,carro,005007-5,Gol GL 1.8 Mi 2p e 4p,VW - VolksWagen,Gasolina,g,1998.0,False,1801300,"R$ 18.013,00",9,2026
34840,carro,035137-7,718 Boxster T 2.0 300cv,Porsche,Gasolina,g,2023.0,False,56127400,"R$ 561.274,00",9,2026
25135,carro,033091-4,Range Rover Vogue SE 4.4 SDV8 Dies. Aut,Land Rover,Diesel,d,2018.0,False,51684600,"R$ 516.846,00",9,2026


In [31]:
df_raw.dtypes

tipo_veiculo             str
codigo_fipe              str
nome_modelo              str
nome_marca               str
nome_combustivel         str
sigla_combustivel        str
ano_modelo           float64
zero_km                 bool
valor_centavos         int64
valor_formatado          str
mes_referencia         int32
ano_referencia         int32
dtype: object

In [32]:
df_raw.columns.tolist()

['tipo_veiculo',
 'codigo_fipe',
 'nome_modelo',
 'nome_marca',
 'nome_combustivel',
 'sigla_combustivel',
 'ano_modelo',
 'zero_km',
 'valor_centavos',
 'valor_formatado',
 'mes_referencia',
 'ano_referencia']

# 4. Pre-Transformation Validation

Before applying transformations, the raw dataset is validated against the structural and semantic assumptions discovered during exploratory profiling.

This step is intended to detect unexpected changes in the source data before any corrective transformation is applied.

## 4.1 Schema Validation

Validate that all expected columns are present and that no unexpected structural changes occurred.

In [45]:
expected_columns = [
    "mes_referencia",
    "ano_referencia",
    "tipo_veiculo",
    "codigo_fipe",
    "nome_marca",
    "nome_modelo",
    "ano_modelo",
    "zero_km",
    "nome_combustivel",
    "sigla_combustivel",
    "valor_centavos",
    "valor_formatado",
]

actual_columns = df_raw.columns.tolist()

missing_columns = set(expected_columns) - set(actual_columns)
unexpected_columns = set(actual_columns) - set(expected_columns)

print("Missing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

Missing columns: set()
Unexpected columns: set()


In [46]:
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

if unexpected_columns:
    raise ValueError(f"Unexpected columns: {unexpected_columns}")

## 4.2 Completeness Validation

Validate the completeness rules defined during profiling.

Relevant rules:

- DQ001 — `codigo_fipe` must not be null;
- DQ002 — all columns except `ano_modelo` must not be null;
- DQ003 — `ano_modelo` may be null only for zero-kilometer vehicles.

## 4.2.1 General Completeness View

In [47]:
null_counts = df_raw.isna().sum()

null_counts[null_counts > 0]

ano_modelo    1931
dtype: int64

## 4.2.2 DQ001 - `codigo_fipe`

In [55]:
# DQ001 — codigo_fipe completeness
codigo_fipe_nulls = df_raw["codigo_fipe"].isna().sum()

if codigo_fipe_nulls > 0:
    raise ValueError(
        f"DQ001 failed: codigo_fipe contains {codigo_fipe_nulls} null values."
    )

## 4.2.3 DQ002 - Other Columns

In [56]:
# DQ002 — completeness for all columns except ano_modelo
required_columns = [
    column
    for column in df_raw.columns
    if column != "ano_modelo"
]

required_null_counts = df_raw[required_columns].isna().sum()

columns_with_nulls = required_null_counts[
    required_null_counts > 0
]

if not columns_with_nulls.empty:
    raise ValueError(
        f"DQ002 failed: unexpected null values found:\n{columns_with_nulls}"
    )

## 4.2.4 DQ003 - `ano_modelo` Null Condition

In [57]:
# DQ003 — conditional nullability of ano_modelo
invalid_null_model_year = df_raw[
    df_raw["ano_modelo"].isna()
    & ~df_raw["zero_km"]
]

invalid_zero_km = df_raw[
    df_raw["zero_km"]
    & df_raw["ano_modelo"].notna()
]

if not invalid_null_model_year.empty:
    raise ValueError(
        "DQ003 failed: ano_modelo is null for non-zero-km vehicles."
    )

if not invalid_zero_km.empty:
    raise ValueError(
        "DQ003 failed: zero-km vehicles contain non-null ano_modelo."
    )

## 4.3 Snapshot Grain Validation

Validate that the expected snapshot grain uniquely identifies each record:

`codigo_fipe + ano_modelo + zero_km + sigla_combustivel`

### 4.3.1 Key Definition

In [59]:
snapshot_key = [
    "codigo_fipe",
    "ano_modelo",
    "zero_km",
    "sigla_combustivel",
]

### 4.3.2 Duplicity Check

In [63]:
duplicate_mask = df_raw.duplicated(
    subset=snapshot_key,
    keep=False
)

In [64]:
duplicate_records = df_raw.loc[duplicate_mask]

In [66]:
duplicate_records.head(20)

,tipo_veiculo,codigo_fipe,nome_modelo,nome_marca,nome_combustivel,sigla_combustivel,ano_modelo,zero_km,valor_centavos,valor_formatado,mes_referencia,ano_referencia


### 4.3.3 Key Validation

In [68]:
if not duplicate_records.empty:
    raise ValueError(
        f"DQ004 failed: {len(duplicate_records)} rows violate the snapshot grain."
    )

## 4.4 Domain Validation

Validate known categorical domains before transformation.

Relevant fields include:

- `tipo_veiculo`;
- `nome_combustivel`;
- `sigla_combustivel`;
- `mes_referencia`;
- `ano_modelo`.

### 4.4.1 `tipo_veiculo`

In [75]:
expected_vehicle_types = {
    "carro",
    "caminhão",
    "moto",
}

actual_vehicle_types = set(df_raw["tipo_veiculo"].dropna().unique())

unexpected_vehicle_types = (
    actual_vehicle_types - expected_vehicle_types
)

if unexpected_vehicle_types:
    raise ValueError(
        f"DQ006 failed: unexpected tipo_veiculo values: "
        f"{unexpected_vehicle_types}"
    )

### 4.4.2 `nome_combustivel`

In [78]:
expected_fuel_names = {
    "Gasolina",
    "Diesel",
    "Flex",
    "Híbrido",
    "Elétrico",
    "Álcool",
    "Gás Natural",
}

actual_fuel_names = set(
    df_raw["nome_combustivel"].dropna().unique()
)

unexpected_fuel_names = (
    actual_fuel_names - expected_fuel_names
)

if unexpected_fuel_names:
    raise ValueError(
        f"Unexpected nome_combustivel values: "
        f"{unexpected_fuel_names}"
    )

### 4.4.3 `sigla_combustivel`

In [79]:
expected_fuel_codes = {
    "g",
    "d",
    "f",
    "h",
    "l",
    "e",
    "n",
}

actual_fuel_codes = set(
    df_raw["sigla_combustivel"].dropna().unique()
)

unexpected_fuel_codes = (
    actual_fuel_codes - expected_fuel_codes
)

if unexpected_fuel_codes:
    raise ValueError(
        f"Unexpected sigla_combustivel values: "
        f"{unexpected_fuel_codes}"
    )

### 4.4.4 `mes_referencia`

In [80]:
invalid_months = df_raw[
    ~df_raw["mes_referencia"].between(1, 12)
]

if not invalid_months.empty:
    raise ValueError(
        f"DQ014 failed: {len(invalid_months)} rows contain "
        "invalid mes_referencia values."
    )

### 4.4.5 `ano_modelo`

In [81]:
reference_year = df_raw["ano_referencia"].iloc[0]

invalid_model_years = df_raw[
    df_raw["ano_modelo"].notna()
    & (
        (df_raw["ano_modelo"] < 1900)
        | (df_raw["ano_modelo"] > reference_year + 1)
    )
]

## 4.5 Referential and Functional Dependency Validation

Validate the deterministic relationships identified during profiling:

- `nome_combustivel` ↔ `sigla_combustivel`;
- `codigo_fipe` → `nome_marca + nome_modelo`;
- `nome_marca + nome_modelo` → `codigo_fipe`.

### 4.5.1 Fuel Name ↔ Fuel Code

In [100]:
# DQ007 — Fuel name/code relationship

expected_fuel_mapping = {
    "Gasolina": "g",
    "Diesel": "d",
    "Flex": "f",
    "Híbrido": "h",
    "Elétrico": "l",
    "Álcool": "e",
    "Gás Natural": "n",
}

expected_fuel_code = df_raw["nome_combustivel"].map(
    expected_fuel_mapping
)

invalid_fuel_mapping = df_raw[
    df_raw["sigla_combustivel"] != expected_fuel_code
]

if not invalid_fuel_mapping.empty:
    raise ValueError(
        f"DQ007 failed: {len(invalid_fuel_mapping)} rows contain "
        "inconsistent fuel mappings."
    )

### 4.5.2 `codigo_fipe → nome_marca + nome_modelo`

In [101]:
# DQ008 — codigo_fipe -> nome_marca + nome_modelo

fipe_code_mapping = (
    df_raw[
        ["codigo_fipe", "nome_marca", "nome_modelo"]
    ]
    .drop_duplicates()
    .groupby("codigo_fipe")
    .size()
)

invalid_fipe_codes = fipe_code_mapping[
    fipe_code_mapping > 1
]

if not invalid_fipe_codes.empty:
    raise ValueError(
        f"DQ008 failed: {len(invalid_fipe_codes)} codigo_fipe values "
        "map to multiple brand-model combinations."
    )

### 4.5.3 `nome_marca + nome_modelo → codigo_fipe`

In [102]:
# DQ009 — nome_marca + nome_modelo -> codigo_fipe

brand_model_mapping = (
    df_raw[
        ["nome_marca", "nome_modelo", "codigo_fipe"]
    ]
    .drop_duplicates()
    .groupby(["nome_marca", "nome_modelo"])
    .size()
)

invalid_brand_models = brand_model_mapping[
    brand_model_mapping > 1
]

if not invalid_brand_models.empty:
    raise ValueError(
        f"DQ009 failed: {len(invalid_brand_models)} brand-model combinations "
        "map to multiple codigo_fipe values."
    )

## 4.6 Monetary Validation

Validate the monetary constraints:

- `valor_centavos > 0`;
- `valor_centavos` and `valor_formatado` represent the same value;
- `valor_formatado` follows the expected Brazilian currency format.

### 4.6.1 DQ010 - Positive Monetary Value

In [103]:
invalid_values = df_raw[
    df_raw['valor_centavos'] <= 0
]

if not invalid_values.empty:
    raise ValueError(
        f"DQ010 failed: {len(invalid_values)} row contain"
        "non-positive valor_centavos values."
    )

### 4.6.2 DQ012 - Brazilian Currency Format

In [108]:
currency_pattern = r"^R\$ \d{1,3}(\.\d{3})*,\d{2}$"

valid_currency_format = df_raw["valor_formatado"].str.match(
    currency_pattern,
    na=False
)

invalid_currency_format = df_raw[
    ~valid_currency_format
]

if not invalid_currency_format.empty:
    raise ValueError(
        f"DQ012 failed: {len(invalid_currency_format)} rows contain "
        "invalid valor_formatado values."
    )

### 4.6.3 `valor_formatado` Conversion

In [112]:
parsed_centavos = (
    df_raw["valor_formatado"]
    .str.replace("R$ ", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype("int64")
)

### 4.6.4 DQ011 - Monetary Consistency

In [113]:
monetary_mismatch = df_raw[
    df_raw["valor_centavos"] != parsed_centavos
]

if not monetary_mismatch.empty:
    raise ValueError(
        f"DQ011 failed: {len(monetary_mismatch)} rows contain "
        "inconsistent monetary representations."
    )

# 5. Data Transformations

Only transformations justified by the exploratory profiling and data quality documentation are applied in this section.

## 5.1 Create Working Copy

In [114]:
df = df_raw.copy()

## 5.2 Brand Name Standardization

`nome_marca` contains capitalization variants representing the same manufacturer.

These variants are standardized to a canonical representation.

In [115]:
brand_mapping = {
    "AGRALE": "Agrale",
    "FIAT": "Fiat",
    "FORD": "Ford",
    "HONDA": "Honda",
    "HYUNDAI": "Hyundai",
    "MERCEDES-BENZ": "Mercedes-Benz",
    "PEUGEOT": "Peugeot",
    "SUZUKI": "Suzuki",
    "VOLVO": "Volvo",
}

df["nome_marca"] = (
    df["nome_marca"]
    .replace(brand_mapping)
)

## 5.3 String Standardization

Textual fields are inspected for technical formatting inconsistencies such as:

- leading whitespace;
- trailing whitespace;
- unintended duplicated spaces.

Semantic content must not be altered.

In [116]:
string_columns = [
    "tipo_veiculo",
    "codigo_fipe",
    "nome_marca",
    "nome_modelo",
    "nome_combustivel",
    "sigla_combustivel",
    "valor_formatado",
]

for column in string_columns:
    df[column] = df[column].str.strip()

## 5.4 Data Type Standardization

Columns are converted to explicit data types appropriate for downstream processing.

The transformations must preserve nullable fields and business semantics.

In [117]:
df["mes_referencia"] = df["mes_referencia"].astype("int64")
df["ano_referencia"] = df["ano_referencia"].astype("int64")
df["ano_modelo"] = df["ano_modelo"].astype("Int64")
df["zero_km"] = df["zero_km"].astype("bool")
df["valor_centavos"] = df["valor_centavos"].astype("int64")

## 5.5 Zero-Kilometer Semantics

Null values in `ano_modelo` associated with `zero_km = True` are preserved.

No model-year imputation is performed because the null state represents a distinct business condition rather than missing information.

## 5.6 Monetary Representation

`valor_centavos` is preserved as the canonical numeric monetary representation.

`valor_formatado` is retained only as a presentation-oriented representation and must remain consistent with `valor_centavos`.

In [118]:
df["valor_centavos"] = df["valor_centavos"].astype("int64")

## 5.7 Column Organization

Columns are reordered according to their analytical role to improve dataset readability and downstream usability.

In [119]:
column_order = [
    "ano_referencia",
    "mes_referencia",
    "tipo_veiculo",
    "codigo_fipe",
    "nome_marca",
    "nome_modelo",
    "ano_modelo",
    "zero_km",
    "nome_combustivel",
    "sigla_combustivel",
    "valor_centavos",
    "valor_formatado",
]

df = df[column_order]

# 6. Post-Transformation Validation

After all transformations are applied, the dataset is validated again.

The objective is to confirm that the transformation process did not introduce inconsistencies and that the processed dataset satisfies the expected data quality contract.

## 6.1 Schema Validation

In [122]:
actual_columns = df.columns.tolist()

missing_columns = set(expected_columns) - set(actual_columns)
unexpected_columns = set(actual_columns) - set(expected_columns)

if missing_columns:
    raise ValueError(
        f"Post-validation failed: missing columns: {missing_columns}"
    )

if unexpected_columns:
    raise ValueError(
        f"Post-validation failed: unexpected columns: {unexpected_columns}"
    )

## 6.2 Completeness Validation

In [123]:
codigo_fipe_nulls = df["codigo_fipe"].isna().sum()

if codigo_fipe_nulls > 0:
    raise ValueError(
        f"DQ001 failed after transformation: "
        f"codigo_fipe contains {codigo_fipe_nulls} null values."
    )

In [124]:
required_columns = [
    column
    for column in df.columns
    if column != "ano_modelo"
]

required_null_counts = df[required_columns].isna().sum()

columns_with_nulls = required_null_counts[
    required_null_counts > 0
]

if not columns_with_nulls.empty:
    raise ValueError(
        "DQ002 failed after transformation: "
        f"unexpected null values found:\n{columns_with_nulls}"
    )

In [125]:
invalid_null_model_year = df[
    df["ano_modelo"].isna()
    & ~df["zero_km"]
]

invalid_zero_km = df[
    df["zero_km"]
    & df["ano_modelo"].notna()
]

if not invalid_null_model_year.empty:
    raise ValueError(
        "DQ003 failed after transformation: "
        "ano_modelo is null for non-zero-km vehicles."
    )

if not invalid_zero_km.empty:
    raise ValueError(
        "DQ003 failed after transformation: "
        "zero-km vehicles contain non-null ano_modelo."
    )

## 6.3 Uniqueness Validation

In [126]:
duplicate_mask = df.duplicated(
    subset=snapshot_key,
    keep=False
)

duplicate_records = df.loc[duplicate_mask]

if not duplicate_records.empty:
    raise ValueError(
        f"DQ004 failed after transformation: "
        f"{len(duplicate_records)} rows violate the snapshot grain."
    )

## 6.4 Domain Validation

### `tipo_veiculo`

In [127]:
actual_vehicle_types = set(
    df["tipo_veiculo"].dropna().unique()
)

unexpected_vehicle_types = (
    actual_vehicle_types - expected_vehicle_types
)

if unexpected_vehicle_types:
    raise ValueError(
        f"DQ006 failed after transformation: "
        f"unexpected tipo_veiculo values: "
        f"{unexpected_vehicle_types}"
    )

### `nome_combustivel`

In [128]:
actual_fuel_names = set(
    df["nome_combustivel"].dropna().unique()
)

unexpected_fuel_names = (
    actual_fuel_names - expected_fuel_names
)

if unexpected_fuel_names:
    raise ValueError(
        f"Unexpected nome_combustivel values after transformation: "
        f"{unexpected_fuel_names}"
    )

In [129]:
actual_fuel_codes = set(
    df["sigla_combustivel"].dropna().unique()
)

unexpected_fuel_codes = (
    actual_fuel_codes - expected_fuel_codes
)

if unexpected_fuel_codes:
    raise ValueError(
        f"Unexpected sigla_combustivel values after transformation: "
        f"{unexpected_fuel_codes}"
    )

### `mes_referencia`

In [130]:
invalid_months = df[
    ~df["mes_referencia"].between(1, 12)
]

if not invalid_months.empty:
    raise ValueError(
        f"DQ014 failed after transformation: "
        f"{len(invalid_months)} rows contain invalid mes_referencia values."
    )

### `ano_referencia`

In [131]:
reference_year = df["ano_referencia"].iloc[0]

invalid_model_years = df[
    df["ano_modelo"].notna()
    & (
        (df["ano_modelo"] < 1900)
        | (df["ano_modelo"] > reference_year + 1)
    )
]

if not invalid_model_years.empty:
    raise ValueError(
        f"Invalid ano_modelo values after transformation: "
        f"{sorted(invalid_model_years['ano_modelo'].unique())}"
    )

## 6.5 Referential Validation

### Fuel Mapping

In [132]:
expected_fuel_code = df["nome_combustivel"].map(
    expected_fuel_mapping
)

invalid_fuel_mapping = df[
    df["sigla_combustivel"] != expected_fuel_code
]

if not invalid_fuel_mapping.empty:
    raise ValueError(
        f"DQ007 failed after transformation: "
        f"{len(invalid_fuel_mapping)} rows contain inconsistent fuel mappings."
    )

### `codigo_fipe → marca + modelo`

In [133]:
fipe_code_mapping = (
    df[
        ["codigo_fipe", "nome_marca", "nome_modelo"]
    ]
    .drop_duplicates()
    .groupby("codigo_fipe")
    .size()
)

invalid_fipe_codes = fipe_code_mapping[
    fipe_code_mapping > 1
]

if not invalid_fipe_codes.empty:
    raise ValueError(
        f"DQ008 failed after transformation: "
        f"{len(invalid_fipe_codes)} codigo_fipe values "
        "map to multiple brand-model combinations."
    )

### `marca + modelo → codigo_fipe`

In [134]:
brand_model_mapping = (
    df[
        ["nome_marca", "nome_modelo", "codigo_fipe"]
    ]
    .drop_duplicates()
    .groupby(["nome_marca", "nome_modelo"])
    .size()
)

invalid_brand_models = brand_model_mapping[
    brand_model_mapping > 1
]

if not invalid_brand_models.empty:
    raise ValueError(
        f"DQ009 failed after transformation: "
        f"{len(invalid_brand_models)} brand-model combinations "
        "map to multiple codigo_fipe values."
    )

## 6.6 Monetary Validation

### DQ010

In [135]:
invalid_values = df[
    df["valor_centavos"] <= 0
]

if not invalid_values.empty:
    raise ValueError(
        f"DQ010 failed after transformation: "
        f"{len(invalid_values)} rows contain non-positive valor_centavos."
    )

### DQ012

In [136]:
valid_currency_format = df["valor_formatado"].str.match(
    currency_pattern,
    na=False
)

invalid_currency_format = df[
    ~valid_currency_format
]

if not invalid_currency_format.empty:
    raise ValueError(
        f"DQ012 failed after transformation: "
        f"{len(invalid_currency_format)} rows contain invalid valor_formatado."
    )

### DQ011

In [137]:
parsed_centavos = (
    df["valor_formatado"]
    .str.replace("R$ ", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype("int64")
)

monetary_mismatch = df[
    df["valor_centavos"] != parsed_centavos
]

if not monetary_mismatch.empty:
    raise ValueError(
        f"DQ011 failed after transformation: "
        f"{len(monetary_mismatch)} rows contain inconsistent monetary values."
    )

## 6.7 Standardization Validation

In [138]:
non_canonical_brands = set(brand_mapping.keys())

remaining_non_canonical_brands = set(
    df["nome_marca"].unique()
) & non_canonical_brands

if remaining_non_canonical_brands:
    raise ValueError(
        "Brand standardization failed. "
        f"Non-canonical values remain: {remaining_non_canonical_brands}"
    )

## 6.8 Data Type Validation

In [142]:
expected_dtypes = {
    "mes_referencia": "int64",
    "ano_referencia": "int64",
    "ano_modelo": "Int64",
    "zero_km": "bool",
    "valor_centavos": "int64",
}

actual_dtypes = {
    column: str(df[column].dtype)
    for column in expected_dtypes
}

dtype_mismatches = {
    column: {
        "expected": expected_dtypes[column],
        "actual": actual_dtypes[column],
    }
    for column in expected_dtypes
    if actual_dtypes[column] != expected_dtypes[column]
}

if dtype_mismatches:
    raise TypeError(
        f"Unexpected dtypes after transformation: "
        f"{dtype_mismatches}"
    )

# 7. Transformation Audit

This section summarizes the effects of the transformation pipeline and provides basic observability over the dataset.

## 7.1 Row Count Reconciliation

In [146]:
raw_rows = len(df_raw)
processed_rows = len(df)

print(f"Raw rows: {raw_rows:,}")
print(f"Processed rows: {processed_rows:,}")

if raw_rows != processed_rows:
    raise ValueError(
        f"Row count mismatch: raw={raw_rows}, processed={processed_rows}"
    )

Raw rows: 51,012
Processed rows: 51,012


## 7.2 Column Count Reconciliation

In [147]:
raw_columns = df_raw.shape[1]
processed_columns = df.shape[1]

print(f"Raw columns: {raw_columns}")
print(f"Processed columns: {processed_columns}")

if raw_columns != processed_columns:
    raise ValueError(
        f"Column count mismatch: raw={raw_columns}, "
        f"processed={processed_columns}"
    )

Raw columns: 12
Processed columns: 12


## 7.3 Changed Fields

In [148]:
brand_changes = (
    df_raw["nome_marca"]
    != df["nome_marca"]
).sum()

print(f"nome_marca values changed: {brand_changes:,}")

nome_marca values changed: 7,209


In [149]:
brand_change_details = (
    pd.DataFrame({
        "raw": df_raw["nome_marca"],
        "processed": df["nome_marca"],
    })
    .query("raw != processed")
    .drop_duplicates()
    .sort_values(["raw", "processed"])
)

brand_change_details

,raw,processed
16,AGRALE,Agrale
7412,FIAT,Fiat
7428,FORD,Ford
17583,HONDA,Honda
18866,HYUNDAI,Hyundai
26176,MERCEDES-BENZ,Mercedes-Benz
33479,PEUGEOT,Peugeot
40321,SUZUKI,Suzuki
44980,VOLVO,Volvo


## 7.4 Unexpected Changes Check

In [159]:
unchanged_value_columns = [
    "mes_referencia",
    "ano_referencia",
    "ano_modelo",
    "zero_km",
    "valor_centavos",
]

unexpected_changes = {}

for column in unchanged_value_columns:
    raw_values = df_raw[column]
    processed_values = df[column]

    both_null = raw_values.isna() & processed_values.isna()

    equal_values = (
        raw_values.eq(processed_values)
        .fillna(False)
        | both_null
    )

    changed_count = (~equal_values).sum()

    if changed_count > 0:
        unexpected_changes[column] = changed_count

if unexpected_changes:
    raise ValueError(
        f"Unexpected value changes detected: {unexpected_changes}"
    )

## 7.5 Expected String Transformations Check

In [160]:
strip_columns = [
    "tipo_veiculo",
    "codigo_fipe",
    "nome_modelo",
    "nome_combustivel",
    "sigla_combustivel",
    "valor_formatado",
]

for column in strip_columns:
    expected_values = df_raw[column].str.strip()

    both_null = expected_values.isna() & df[column].isna()

    equal_values = (
        expected_values.eq(df[column])
        .fillna(False)
        | both_null
    )

    if not equal_values.all():
        raise ValueError(
            f"Unexpected transformation detected in {column}."
        )

## 7.6 Brand Standardization Audit

In [161]:
expected_brand = (
    df_raw["nome_marca"]
    .str.strip()
    .replace(brand_mapping)
)

both_null = expected_brand.isna() & df["nome_marca"].isna()

brand_match = (
    expected_brand.eq(df["nome_marca"])
    .fillna(False)
    | both_null
)

if not brand_match.all():
    raise ValueError(
        "Unexpected transformation detected in nome_marca."
    )

## 7.7 Audit Summary

In [165]:
brand_changes = (
    df_raw["nome_marca"]
    != df["nome_marca"]
).sum()

model_name_changes = (
    df_raw["nome_modelo"]
    != df["nome_modelo"]
).sum()

audit_summary = pd.Series({
    "raw_rows": len(df_raw),
    "processed_rows": len(df),
    "raw_columns": df_raw.shape[1],
    "processed_columns": df.shape[1],
    "brand_values_changed": brand_changes,
    "model_name_values_changed": model_name_changes,
})

audit_summary

raw_rows                     51012
processed_rows               51012
raw_columns                     12
processed_columns               12
brand_values_changed          7209
model_name_values_changed      393
dtype: int64

In [ ]:
strip_change_counts = {}

for column in strip_columns:
    strip_change_counts[column] = (
        df_raw[column] != df[column]
    ).sum()

pd.Series(strip_change_counts).sort_values(ascending=False)

nome_modelo          393
tipo_veiculo           0
codigo_fipe            0
nome_combustivel       0
sigla_combustivel      0
valor_formatado        0
dtype: int64

# 8. Processed Data Export

The validated transformed dataset is persisted in a columnar format for downstream analytical processing.

Parquet is used because it preserves data types and provides efficient storage and query performance.

In [169]:
output_file = (
    PROCESSED_DATA_DIR
    / "fipex_prices_2026_09.parquet"
)

df.to_parquet(
    output_file,
    index=False,
)

In [170]:
if not output_file.exists():
    raise FileNotFoundError(
        f"Processed file was not created: {output_file}"
    )

In [171]:
print(f"Processed dataset saved to: {output_file}")

Processed dataset saved to: e:\VSCODE Files\Projects\01_automotive_market_data_analysis\data\processed\fipex_prices_2026_09.parquet


## 8.1 Read-Back Validation

In [172]:
df_check = pd.read_parquet(output_file)

In [173]:
if df_check.shape != df.shape:
    raise ValueError(
        f"Export validation failed: "
        f"expected shape {df.shape}, got {df_check.shape}"
    )

In [174]:
if list(df_check.columns) != list(df.columns):
    raise ValueError(
        "Export validation failed: column order mismatch."
    )

# 9. Transformation Summary

The FIPE raw snapshot was successfully processed and validated.

The pipeline:

- preserved the raw source unchanged;
- validated structural and semantic constraints before transformation;
- standardized known brand-name inconsistencies;
- removed leading and trailing whitespace from selected textual fields;
- standardized data types;
- preserved zero-kilometer model-year semantics;
- revalidated the complete data quality contract;
- reconciled row and column counts;
- audited expected field-level changes;
- persisted the processed dataset in Parquet format.

The processed dataset is ready for downstream SQL-based analytical modeling.